# ⚽🌍 Pipeline Multiagente — Análisis de Partidos de la Copa del Mundo

Pipeline con 4 agentes especializados:
1. **Normalizador** – limpieza, imputación, feature engineering y codificación
2. **Entrenador** – Random Forest Classifier (predicción de resultado)
3. **Comunicador** – reporte de métricas en lenguaje natural
4. **Chatbot** – interfaz conversacional con acceso directo al dataset histórico

## ⚙️ 0. Instalación de dependencias

In [ ]:
import subprocess, sys

for pkg in ["pandas", "numpy", "scikit-learn", "requests", "matplotlib", "seaborn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ Dependencias listas.")

## 📦 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import warnings
import requests
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
print("✅ Imports correctos.")

## 📂 2. Carga del Dataset

In [ ]:
from google.colab import files

print("📂 Seleccioná el archivo wc_all_matches.csv:")
uploaded = files.upload()

CSV_FILE = list(uploaded.keys())[0]
print(f"✅ Archivo '{CSV_FILE}' cargado.")

In [ ]:
df_wc = pd.read_csv(CSV_FILE)
print(f"✅ Dataset cargado → {df_wc.shape[0]} partidos | {df_wc['year'].min()}–{df_wc['year'].max()}")
print(f"   Columnas: {list(df_wc.columns)}")
df_wc.head(8)

## 🤖 Agente 1 — Normalizador

Limpia, crea features derivadas, codifica categóricas y escala variables numéricas.

In [ ]:
class NormalizerAgent:
    """Limpieza, feature engineering, codificación y escalado del dataset Mundial."""

    def __init__(self):
        self.le_stage   = LabelEncoder()
        self.le_team1   = LabelEncoder()
        self.le_team2   = LabelEncoder()
        self.le_country = LabelEncoder()
        self.scaler     = MinMaxScaler()

    def process(self, df: pd.DataFrame):
        print("🤖 [Agente 1 - Normalizador]: Iniciando normalización...")
        df_clean = df.copy()

        # 1. Limpieza
        nulos_antes = df_clean.isnull().sum().sum()
        df_clean['notes'] = df_clean['notes'].fillna('Sin nota')
        df_clean['date']  = pd.to_datetime(df_clean['date'], errors='coerce')
        df_clean = df_clean.dropna(subset=['score1', 'score2'])
        print(f"   ✔ Nulos: {nulos_antes} → {df_clean.isnull().sum().sum()}")

        # 2. Feature engineering
        df_clean['total_goles']          = df_clean['score1'] + df_clean['score2']
        df_clean['diferencia_goles']     = df_clean['score1'] - df_clean['score2']
        df_clean['es_empate']            = (df_clean['score1'] == df_clean['score2']).astype(int)
        df_clean['partido_alto_scoring'] = (df_clean['total_goles'] > 3).astype(int)
        df_clean['decada']               = (df_clean['year'] // 10) * 10
        df_clean['resultado'] = df_clean['diferencia_goles'].apply(
            lambda x: 'victoria' if x > 0 else ('derrota' if x < 0 else 'empate')
        )
        df_clean['resultado_enc'] = df_clean['resultado'].map({'victoria':2,'empate':1,'derrota':0})
        print("   ✔ Features: total_goles, diferencia_goles, es_empate, decada, resultado")

        # 3. Codificación
        df_clean['stage_enc']   = self.le_stage.fit_transform(df_clean['stage'])
        df_clean['team1_enc']   = self.le_team1.fit_transform(df_clean['team1'])
        df_clean['team2_enc']   = self.le_team2.fit_transform(df_clean['team2'])
        df_clean['country_enc'] = self.le_country.fit_transform(df_clean['country'])
        print("   ✔ Codificación: stage, team1, team2, country → _enc")

        # 4. Escalado
        cols_num = ['year','score1','score2','total_goles','diferencia_goles']
        scaled   = self.scaler.fit_transform(df_clean[cols_num])
        for i, col in enumerate(cols_num):
            df_clean[col + '_scaled'] = scaled[:, i]
        print("   ✔ Escalado MinMax aplicado")

        print(f"✅ [Agente 1]: Dataset limpio → {df_clean.shape[0]} filas × {df_clean.shape[1]} cols")
        return df_clean


TARGET_COL = 'resultado_enc'

agente_normalizador = NormalizerAgent()
df_limpio = agente_normalizador.process(df_wc)
df_limpio[['year','stage','team1','score1','score2','team2','total_goles','resultado','decada']].head(8)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df_limpio['total_goles'].hist(ax=axes[0], bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de Goles por Partido')

df_limpio['resultado'].value_counts().plot(kind='bar', ax=axes[1],
    color=['#2ecc71','#e74c3c','#95a5a6'], edgecolor='white')
axes[1].set_title('Resultados (perspectiva team1)')
axes[1].tick_params(axis='x', rotation=0)

df_limpio.groupby('decada')['total_goles'].mean().plot(kind='bar', ax=axes[2],
    color='coral', edgecolor='white')
axes[2].set_title('Promedio Goles por Década')

plt.suptitle('Dataset Copa del Mundo 1930–2022', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 🤖 Agente 2 — Entrenador

In [ ]:
class TrainerAgent:
    """Entrena Random Forest Classifier con validación cruzada estratificada."""

    FEATURES = [
        'year_scaled','stage_enc','team1_enc','team2_enc',
        'country_enc','decada','es_empate','partido_alto_scoring'
    ]

    def __init__(self):
        self.model    = RandomForestClassifier(n_estimators=200, max_depth=15,
                                               random_state=42, n_jobs=-1)
        self.metricas = {}

    def process(self, df: pd.DataFrame, target_column: str):
        print("🤖 [Agente 2 - Entrenador]: Iniciando entrenamiento...")
        X = df[self.FEATURES]
        y = df[target_column]

        cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(self.model, X, y, cv=cv, scoring='accuracy')
        print(f"   ✔ CV (5-fold): {scores.mean():.4f} ± {scores.std():.4f}")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42)
        self.model.fit(X_train, y_train)
        y_pred = self.model.predict(X_test)

        self.metricas = {
            'accuracy_cv_media': round(float(scores.mean()), 4),
            'accuracy_cv_std'  : round(float(scores.std()),  4),
            'accuracy_test'    : round(float(accuracy_score(y_test, y_pred)), 4),
            'y_test'           : y_test.tolist(),
            'y_pred'           : y_pred.tolist(),
            'confusion_matrix' : confusion_matrix(y_test, y_pred).tolist(),
        }
        print(f"✅ [Agente 2]: Accuracy test = {self.metricas['accuracy_test']:.4f}")
        return self.model, self.metricas


agente_entrenador = TrainerAgent()
modelo_entrenado, metricas_modelo = agente_entrenador.process(df_limpio, TARGET_COL)

print("\n📊 Reporte de clasificación:")
print(classification_report(metricas_modelo['y_test'], metricas_modelo['y_pred'],
      target_names=['Derrota','Empate','Victoria']))

# Matriz de confusión
cm = np.array(metricas_modelo['confusion_matrix'])
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Derrota','Empate','Victoria'],
            yticklabels=['Derrota','Empate','Victoria'], ax=ax)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión — Random Forest')
plt.tight_layout(); plt.show()

## 🤖 Agente 3 — Comunicador

In [ ]:
class CommunicatorAgent:
    """Genera reporte ejecutivo con métricas e insights del dataset."""

    def process(self, metricas: dict, df_original: pd.DataFrame):
        print("\n🤖 [Agente 3 - Comunicador]: Generando reporte...\n")

        acc_cv   = metricas['accuracy_cv_media'] * 100
        acc_test = metricas['accuracy_test'] * 100
        total_g  = int(df_original['score1'].sum() + df_original['score2'].sum())
        prom_g   = round((df_original['score1'] + df_original['score2']).mean(), 2)
        anios    = f"{int(df_original['year'].min())} – {int(df_original['year'].max())}"
        top_team = df_original.groupby('team1')['score1'].sum().sort_values(ascending=False).index[0]

        r  = "⚽ " + "="*60 + "\n"
        r += "   REPORTE — COPA DEL MUNDO | Pipeline Multiagente\n"
        r += "="*62 + "\n\n"
        r += f"📂 Período:              {anios}\n"
        r += f"   Partidos analizados:  {len(df_original)}\n"
        r += f"   Total de goles:       {total_g}\n"
        r += f"   Promedio goles/ptdo:  {prom_g}\n"
        r += f"   Equipo más goleador:  {top_team}\n\n"
        r += f"🎯 Modelo:               Random Forest Classifier\n"
        r += f"   Accuracy CV (5-fold): {acc_cv:.2f}% ± {metricas['accuracy_cv_std']*100:.2f}%\n"
        r += f"   Accuracy test:        {acc_test:.2f}%\n\n"
        r += "💡 El modelo predice victoria/empate/derrota con esos valores.\n"
        r += "   El fútbol es inherentemente impredecible, esto es esperable.\n"
        r += "\n🚀 Pipeline completado exitosamente.\n" + "="*62

        print(r)
        return r


agente_comunicador = CommunicatorAgent()
reporte_final = agente_comunicador.process(metricas_modelo, df_wc)

## 🤖 Agente 4 — Chatbot con acceso al dataset histórico

Este agente responde preguntas sobre partidos reales gracias a una capa de **búsqueda directa en el dataset** combinada con **Mistral AI** como motor de lenguaje natural.

**Ejemplos de preguntas:**
- *¿Cómo le fue a Argentina en el mundial 2022?*
- *¿Cuál fue el resultado de la final del mundial 1986?*
- *¿Qué partidos jugó Brasil en 1970?*
- *¿Cuántos goles hizo Alemania en la semifinal 2014?*
- *¿Quién ganó el mundial de 1998?*

In [ ]:
class ChatbotAgent:
    """
    Agente 4 — Chatbot con búsqueda en el dataset del Mundial + Mistral AI.

    Flujo por pregunta:
      1. Extrae entidades de la pregunta (año, equipos, fase) con regex.
      2. Busca partidos que coincidan en el dataset.
      3. Formatea los resultados encontrados como contexto adicional.
      4. Llama a Mistral con ese contexto enriquecido.
    """

    MISTRAL_URL = "https://api.mistral.ai/v1/chat/completions"

    STAGES = [
        "group stage", "round of 16", "quarter-final", "semi-final",
        "final round", "final", "3rd place"
    ]

    def __init__(self, df: pd.DataFrame, api_key: str = "TU_API_KEY_MISTRAL"):
        self.df      = df.copy()
        self.api_key = api_key
        self.df['team1_lower']   = self.df['team1'].str.lower()
        self.df['team2_lower']   = self.df['team2'].str.lower()
        self.df['stage_lower']   = self.df['stage'].str.lower()
        self.df['country_lower'] = self.df['country'].str.lower()
        self.historial           = []

    # ── Extracción de entidades ───────────────────────────────────────────────
    def _extraer_anio(self, texto: str):
        """Extrae el primer año de 4 dígitos entre 1930 y 2022."""
        matches = re.findall(r'\b(19[3-9]\d|200\d|201\d|2022)\b', texto)
        return int(matches[0]) if matches else None

    def _extraer_equipos(self, texto: str):
        """Busca nombres de equipos del dataset dentro del texto."""
        texto_lower = texto.lower()
        equipos = set(self.df['team1'].unique()) | set(self.df['team2'].unique())
        encontrados = []
        for eq in equipos:
            if re.search(rf'\b{re.escape(eq.lower())}\b', texto_lower):
                encontrados.append(eq)
        return encontrados

    def _extraer_fase(self, texto: str):
        """Detecta la fase mencionada en la pregunta."""
        texto_lower = texto.lower()
        for fase in self.STAGES:
            if fase in texto_lower:
                return fase
        aliases = {
            'final'         : 'final',
            'semifinal'     : 'semi-final',
            'cuartos'       : 'quarter-final',
            'octavos'       : 'round of 16',
            'grupos'        : 'group stage',
            'fase de grupos': 'group stage',
            'tercer puesto' : '3rd place',
            'tercer'        : '3rd place',
            'tercero'       : '3rd place',
        }
        for alias, fase in aliases.items():
            if alias in texto_lower:
                return fase
        return None

    # ── Búsqueda en el dataset ────────────────────────────────────────────────
    def _buscar_partidos(self, anio, equipos, fase):
        """Filtra el dataset según año, equipos y/o fase detectados."""
        df = self.df.copy()
        filtrado = False

        if anio:
            df = df[df['year'] == anio]
            filtrado = True
        if equipos:
            mask = pd.Series([False] * len(df), index=df.index)
            for eq in equipos:
                eq_lower = eq.lower()
                mask = mask | (df['team1_lower'] == eq_lower) | (df['team2_lower'] == eq_lower)
            df = df[mask]
            filtrado = True
        if fase:
            df = df[df['stage_lower'].str.contains(fase, na=False)]
            filtrado = True

        return df if filtrado else pd.DataFrame()

    def _formatear_partidos(self, df_result: pd.DataFrame) -> str:
        """Convierte filas del dataset en texto legible para el contexto."""
        if df_result.empty:
            return ""

        lineas = [f"Se encontraron {len(df_result)} partido(s) en el dataset:\n"]
        for _, row in df_result.iterrows():
            score   = f"{int(row['score1'])}–{int(row['score2'])}"
            ganador = row['team1'] if row['score1'] > row['score2'] else (
                      row['team2'] if row['score2'] > row['score1'] else 'Empate')
            nota    = f" ({row['notes']})" if row['notes'] not in ['', 'Sin nota'] else ''
            lineas.append(
                f"  • {int(row['year'])} | {row['stage']:20s} | "
                f"{row['team1']} {score} {row['team2']}"
                f" — {ganador}{nota}"
            )
        return "\n".join(lineas)

    # ── Llamada a Mistral ─────────────────────────────────────────────────────
    def _llamar_mistral(self, pregunta: str, contexto_dataset: str) -> str:
        """Envía la pregunta a Mistral con el contexto del dataset enriquecido."""
        system_prompt = (
            "Eres un asistente experto en la historia de la Copa del Mundo de Fútbol (1930–2022). "
            f"Tienes acceso a un dataset con {len(self.df)} partidos oficiales. "
            f"El dataset cubre desde {int(self.df['year'].min())} hasta {int(self.df['year'].max())}. "
            "Cuando el usuario pregunta por un partido o resultado, recibirás los datos exactos del dataset. "
            "Usá esa información para dar una respuesta precisa y detallada. "
            "Si no hay datos en el contexto, respondé con tu conocimiento general pero aclaralo. "
            "Respondé siempre en español, con datos concretos y tono amigable."
        )
        if contexto_dataset:
            system_prompt += f"\n\nDATOS DEL DATASET PARA ESTA CONSULTA:\n{contexto_dataset}"

        headers = {
            "Content-Type" : "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }
        payload = {
            "model"      : "mistral-small-latest",
            "messages"   : [{"role": "system", "content": system_prompt}] + self.historial,
            "temperature": 0.3,
        }
        resp = requests.post(self.MISTRAL_URL, headers=headers, json=payload, timeout=30)
        if resp.status_code == 200:
            return resp.json()["choices"][0]["message"]["content"]
        else:
            return f"❌ Error de la API ({resp.status_code}): {resp.text}"

    # ── Sesión principal ──────────────────────────────────────────────────────
    def iniciar_sesion(self):
        print("\n" + "="*55)
        print("🤖 Agente 4 — Chatbot | Copa del Mundo 1930–2022")
        print("="*55)

        if self.api_key == "TU_API_KEY_MISTRAL":
            print("⚠️  Reemplazá MISTRAL_API_KEY con tu clave real y volvé a ejecutar.")
            return

        print("Preguntame sobre partidos, resultados o estadísticas.")
        print("Escribí 'salir' para terminar.\n")

        while True:
            pregunta = input("👤 Vos: ").strip()
            if not pregunta:
                continue
            if pregunta.lower() in ['salir', 'exit', 'quit']:
                print("🤖 ¡Hasta la próxima Copa del Mundo!")
                break

            self.historial.append({"role": "user", "content": pregunta})

            anio    = self._extraer_anio(pregunta)
            equipos = self._extraer_equipos(pregunta)
            fase    = self._extraer_fase(pregunta)

            df_resultados    = self._buscar_partidos(anio, equipos, fase)
            contexto_dataset = self._formatear_partidos(df_resultados)

            if contexto_dataset:
                print(f"\n   🔍 Encontré {len(df_resultados)} partido(s) en el dataset.")

            try:
                print("   ⏳ Consultando a Mistral...")
                respuesta = self._llamar_mistral(pregunta, contexto_dataset)
                print(f"\n🤖 Agente 4: {respuesta}\n")
                self.historial.append({"role": "assistant", "content": respuesta})

            except requests.exceptions.Timeout:
                print("❌ Timeout. Intentá de nuevo.")
            except Exception as e:
                print(f"❌ Error: {e}")

In [ ]:
# ─── Ingresá tu API key de Mistral ───────────────────────────────────────────
# Obtenela en: https://console.mistral.ai/
MISTRAL_API_KEY = "TU_API_KEY_MISTRAL"  # ← reemplazá con tu clave

agente_chatbot = ChatbotAgent(df=df_wc, api_key=MISTRAL_API_KEY)
agente_chatbot.iniciar_sesion()

## 🧪 Modo debug — Búsqueda sin API

Si no tenés la API key de Mistral todavía, podés probar la capa de búsqueda directamente.

In [ ]:
def buscar_sin_api(df_wc: pd.DataFrame, pregunta: str):
    """
    Prueba la capa de búsqueda en el dataset sin llamar a ninguna API.
    Útil para verificar que la extracción de entidades funciona bien.
    """
    bot_test = ChatbotAgent(df=df_wc, api_key="TEST")

    anio    = bot_test._extraer_anio(pregunta)
    equipos = bot_test._extraer_equipos(pregunta)
    fase    = bot_test._extraer_fase(pregunta)

    print(f"📝 Pregunta:  '{pregunta}'")
    print(f"   Año:       {anio}")
    print(f"   Equipos:   {equipos}")
    print(f"   Fase:      {fase}")

    df_r = bot_test._buscar_partidos(anio, equipos, fase)
    ctx  = bot_test._formatear_partidos(df_r)

    if ctx:
        print(f"\n{ctx}")
    else:
        print("\n⚠️ No se encontraron partidos para esa consulta.")
    print()


# ── Probá estas consultas ─────────────────────────────────────────────────────
buscar_sin_api(df_wc, "¿Cuál fue el resultado de la final del mundial 1986?")
buscar_sin_api(df_wc, "¿Qué partidos jugó Brasil en el mundial de 1970?")
buscar_sin_api(df_wc, "¿Cómo le fue a Argentina en el mundial 2022?")
buscar_sin_api(df_wc, "¿Cuántos goles metió Alemania en la semifinal 2014 contra Brasil?")
buscar_sin_api(df_wc, "resultados de las semifinales del mundial 1998")

## 📊 Resumen del Pipeline

In [ ]:
print("="*62)
print("   RESUMEN DEL PIPELINE MULTIAGENTE — COPA DEL MUNDO")
print("="*62)
print(f"  Dataset:              {df_wc.shape[0]} partidos | "
      f"{df_wc['year'].min()}–{df_wc['year'].max()}")
print(f"  Dataset normalizado:  {df_limpio.shape[0]} filas × {df_limpio.shape[1]} cols")
print(f"  Modelo:               Random Forest Classifier")
print(f"  Accuracy CV (5-fold): {metricas_modelo['accuracy_cv_media']:.4f} "
      f"± {metricas_modelo['accuracy_cv_std']:.4f}")
print(f"  Accuracy test:        {metricas_modelo['accuracy_test']:.4f}")
print()
print("  Agente 4 — capacidades de búsqueda:")
print(f"  • Extrae año (1930–2022) de la pregunta con regex")
print(f"  • Detecta {len(set(df_wc['team1'].unique()) | set(df_wc['team2'].unique()))} equipos únicos")
print(f"  • Reconoce {len(df_wc['stage'].unique())} fases diferentes del torneo")
print(f"  • Inyecta resultados reales como contexto a Mistral")
print("="*62)